# Markov Chain Monte Carlo (MCMC) & Introduction to Pyro

**Date**: 2025/02/11

Pyro is a flexible, scalable deep probabilistic programming library built on PyTorch. It enables us to combine
the power of deep learning with Bayesian inference by providing tools for specifying complex probabilistic models
and performing inference using advanced techniques such as Markov Chain Monte Carlo (MCMC), Variational Inference,
and more. In this notebook, we explore several sampling methods including:

- **Rejection Sampling**
- **Metropolis-Hastings Algorithm**
- **Langevin Sampling (MALA)**
- **Hamiltonian Monte Carlo (HMC)**
- **MCMC with the No-U-Turn Sampler (NUTS)**

In [ ]:
import torch
import pyro
import pyro.distributions as dist
import pyro.infer.mcmc as mcmc
import matplotlib.pyplot as plt
import seaborn as sns
import time

# Set a random seed for reproducibility
torch.manual_seed(42)

In [ ]:
def time_sampling(sampler_fn, *args, **kwargs):
    start_time = time.time()
    samples = sampler_fn(*args, **kwargs)
    elapsed_time = time.time() - start_time
    return samples, elapsed_time

## 1. Rejection Sampling

![](https://www.researchgate.net/profile/Eric-Postma/publication/23264061/figure/fig1/AS:667209948987410@1536086656432/llustration-of-the-rejection-sampling-procedure-The-KDE-modeimodei-f-x-is-represented.png)

**Rejection sampling** is a fundamental technique in Monte Carlo methods, enabling sampling from a complex **target distribution** $\pi(x)$ using a **proposal distribution** $q(x)$, from which drawing samples is computationally easier. The key to rejection sampling is ensuring that $q(x)$ sufficiently **dominates** $\pi(x)$ through a multiplicative bound:

$$
\pi(x) \leq M q(x), \quad \forall x.
$$

This condition ensures that we can construct a valid sampling rule where each proposal is accepted with probability:

$$
P(\text{accept } x) = \frac{\pi(x)}{M q(x)}
$$

### **1.1 Correctness**

To show that rejection sampling produces samples from $\pi(x)$, consider the probability density function of the accepted samples.

#### **Step 1: Joint Density of Sampling and Accepting**
- The proposal distribution $q(x)$ defines the probability of drawing $x$.
- The probability of acceptance given $x$ is $\frac{\pi(x)}{M q(x)}$.
- Thus, the joint probability of drawing $x$ and accepting it is:

  $$
  P(\text{sample } x \text{ and accept}) = q(x) \cdot \frac{\pi(x)}{M q(x)} = \frac{\pi(x)}{M}.
  $$

#### **Step 2: Normalizing to Obtain the Target Distribution**
- The probability of accepting **any** sample is:

  $$
  P(\text{accept}) = \int P(\text{sample } x \text{ and accept}) dx = \int \frac{\pi(x)}{M} dx = \frac{1}{M} \int \pi(x) dx = \frac{1}{M}.
  $$

- Given that the normalization ensures total probability sums to 1, the density of the accepted samples is:

  $$
  P_{\text{accepted}}(x) = \frac{P(\text{sample } x \text{ and accept})}{P(\text{accept})} = \frac{\frac{\pi(x)}{M}}{\frac{1}{M}} = \pi(x).
  $$

Thus, rejection sampling correctly generates samples from $\pi(x)$.

### **1.2 Efficiency and Expected Number of Samples**
The **efficiency** of rejection sampling depends on the probability of acceptance:

$$
P(\text{accept}) = \frac{1}{M}.
$$

Since each sample is accepted independently with probability $\frac{1}{M}$, the number of draws from $q(x)$ required to obtain one accepted sample follows a geometric distribution with expectation:

$$
E[N] = M.
$$

- If $M$ is **small** (i.e., $q(x)$ closely approximates $\pi(x)$), the method is **efficient**.
- If $M$ is **large** (i.e., $q(x)$ is a poor fit for $\pi(x)$), many samples are rejected, making the method inefficient.


### **1.3 Choosing an Optimal Proposal Distribution**
The **efficiency** of rejection sampling depends on how well $q(x)$ matches $\pi(x)$. The best choice of $q(x)$ minimizes $M$, which means:

$$
M = \sup_x \frac{\pi(x)}{q(x)}
$$

For **optimal efficiency**:
- $q(x)$ should have a shape similar to $\pi(x)$ to minimize rejection.
- A common choice is a **majorizing distribution**, such as a normal distribution covering a target density with heavy tails.

### **1.4 Advantages and Limitations**
#### **Advantages:**
- **Simple implementation** requiring only the ability to sample from $q(x)$.
- **Does not require normalization** of $\pi(x)$, making it useful for posterior sampling in Bayesian inference.

#### **Limitations:**
- **Inefficiency in high dimensions**: As the dimension $d$ increases, the required $M$ grows exponentially, leading to excessive rejection.
- **Requires finding an appropriate $q(x)$**: If $q(x)$ is not well chosen, the method is computationally impractical.

In [ ]:
def rejection_sampling(target, proposal, M, num_samples=1000):
    """
    Perform rejection sampling to draw samples from the target distribution.
    
    Args:
        target (pyro.distributions.Distribution): The target distribution $\pi(x)$.
        proposal (pyro.distributions.Distribution): The proposal distribution $q(x)$.
        M (float): A constant satisfying $\pi(x) \leq M \, q(x)$ for all x.
        num_samples (int): Number of samples to generate.
        
    Returns:
        list: Samples drawn from the target distribution.
    """
    samples = []
    while len(samples) < num_samples:
        # Sample x from the proposal distribution q(x)
        x = proposal.sample()
        # Draw a uniform random number u from U(0,1)
        u = torch.rand(1)
        # Compute the acceptance ratio: exp(log(pi(x))) / (M * exp(log(q(x))))
        acceptance_ratio = torch.exp(target.log_prob(x)) / (M * torch.exp(proposal.log_prob(x)))
        # Accept the sample if u < acceptance_ratio
        if u < acceptance_ratio:
            samples.append(x.item())
    return samples

# Define the target and proposal distributions.
target = dist.Normal(0, 1)   # Target: Standard Normal, $\pi(x) = \mathcal{N}(0,1)$
proposal = dist.Normal(0, 2) # Proposal: Normal with mean 0 and std 2, $q(x)$
M = 1.5  # Scaling constant ensuring $\pi(x) \leq M \, q(x)$
num_samples = 1000

samples, rs_time = time_sampling(rejection_sampling, target, proposal, M, num_samples=num_samples)
print(f"Rejection sampling took {rs_time:.2f} seconds for {num_samples} samples.")

In [ ]:
# Plot the histogram of samples with a kernel density estimate (KDE)
sns.histplot(samples, kde=True)
plt.title("Rejection Sampling")
plt.xlabel("x")
plt.ylabel("Frequency")
plt.show()

## 2. Metropolis-Hastings Algorithm

The **Metropolis-Hastings (MH) algorithm** is a **Markov Chain Monte Carlo (MCMC)** method designed to generate samples from a target distribution $\pi(x)$ when direct sampling is difficult. Unlike rejection sampling, which generates independent samples and discards many of them, MH constructs a **Markov chain** whose stationary distribution is $\pi(x)$. Over time, the samples approximate the desired distribution.

### **2.1 Key Idea: Constructing a Markov Chain**
The MH algorithm creates a sequence $x_1, x_2, \dots, x_n$ of dependent samples forming a **Markov chain**. This sequence converges to the desired distribution $\pi(x)$ under mild conditions.

The key principle behind MH is **balance**:
- If the chain is in state $x$, a new candidate state $x'$ is proposed from a distribution $q(x' \mid x)$.
- The move is accepted with probability:

  $$
  \alpha(x, x') = \min\left\{1, \frac{\pi(x') q(x \mid x')}{\pi(x) q(x' \mid x)}\right\}.
  $$

  - If $\alpha(x, x') = 1$, the move is always accepted.
  - Otherwise, the move is accepted with probability $\alpha(x, x')$, ensuring correct weighting.

This construction **ensures detailed balance**, a fundamental requirement for the chain to converge to $\pi(x)$.

### **2.2 Why Metropolis-Hastings Works**
The Metropolis-Hastings algorithm satisfies two key properties that guarantee convergence to $\pi(x)$:

#### **(a) Ergodicity**
- The Markov chain **explores the full space**, meaning every state has a nonzero probability of being visited.
- This ensures that, given enough time, the chain will sample from the entire target distribution.

#### **(b) Detailed Balance**
- The **transition probability** from $x$ to $x'$ satisfies:

  $$
  \pi(x) P(x \to x') = \pi(x') P(x' \to x).
  $$

- This ensures that $\pi(x)$ is the **stationary distribution**, meaning once the chain reaches equilibrium, it stays there.

Together, these properties mean that after a **burn-in** period, the generated samples approximate $\pi(x)$.

### **2.3 Efficiency and Mixing Properties**
The efficiency of MH depends on:
- **The choice of proposal distribution $q(x' \mid x)$**:
  - If too narrow, the chain moves slowly (high autocorrelation).
  - If too wide, proposals are often rejected.
- **The acceptance rate**:
  - Ideally, around **25-50%** for high-dimensional problems.
  - If too high ($\approx 100\%$), the chain does not explore well.
  - If too low ($\approx 0\%$), most moves are rejected.

#### **Choosing a Good Proposal**
For a Gaussian target $\mathcal{N}(0,1)$, a natural choice is a **Gaussian random walk proposal**:

$$
q(x' \mid x) = \mathcal{N}(x, \sigma^2).
$$

which is used in our implementation:

```python
proposal_fn = lambda x: x + torch.randn(1) * 0.5
```

Here, **0.5 controls step size**—smaller values ensure smoother exploration, while larger values might cause rejection.


### **2.4 Advantages and Limitations**
#### **Advantages**
- **Scales better than rejection sampling**: No need for an explicit bounding constant $M$.  
- **Works in high dimensions**: Unlike rejection sampling, which becomes impractical in large spaces.  
- **Adapts to complex distributions**: Works even when $\pi(x)$ is multi-modal or unnormalized.  

#### **Limitations**
- **Correlated samples**: Unlike rejection sampling, each sample depends on the previous one.  
- **Tuning is required**: Poor choices of $q(x' \mid x)$ lead to slow mixing.  
- **Needs burn-in period**: Early samples might not represent $\pi(x)$, so initial steps are often discarded.  

In [ ]:
def metropolis_hastings(target, proposal_fn, num_samples=1000):
    """
    Perform sampling using the Metropolis-Hastings algorithm.
    
    Args:
        target (pyro.distributions.Distribution): The target distribution $\pi(x)$.
        proposal_fn (function): A function that proposes a new sample given the current sample.
        num_samples (int): Number of samples to generate.
        
    Returns:
        list: Samples approximating the target distribution.
    """
    # Initialize the chain at 0.0
    x = torch.tensor(0.0)
    samples = []
    
    for _ in range(num_samples):
        # Propose a new state using the provided proposal function
        x_new = proposal_fn(x)
        # For a symmetric proposal, acceptance probability:
        # $\alpha(x, x') = \min\{1, \pi(x')/\pi(x)\}$
        acceptance_prob = min(1, (torch.exp(target.log_prob(x_new)) / torch.exp(target.log_prob(x))).item())
        # Draw a random number to decide whether to accept the proposal
        if torch.rand(1) < acceptance_prob:
            x = x_new  # Accept the new state
        samples.append(x.item())
    
    return samples

# Define a simple random walk proposal: current state + Gaussian noise
proposal_fn = lambda x: x + torch.randn(1) * 0.5
num_samples = 1000

# Generate samples using the Metropolis-Hastings algorithm
samples, mh_time = time_sampling(metropolis_hastings, target, proposal_fn, num_samples=num_samples)
print(f"Metropolis-Hastings algorithm took {mh_time:.2f} seconds for {num_samples} samples.")

In [ ]:
# Plot the histogram of the Metropolis-Hastings samples with KDE
sns.histplot(samples, kde=True)
plt.title("Metropolis-Hastings Sampling")
plt.xlabel("x")
plt.ylabel("Frequency")
plt.show()

## 3. Langevin Sampling (MALA)

The **Metropolis-Adjusted Langevin Algorithm (MALA)** is an **MCMC method** that improves upon the **Metropolis-Hastings (MH) algorithm** by incorporating **gradient information** of the log-probability function $ \log \pi(x) $ to guide the sampling process. 

This allows for **more efficient exploration** of high-dimensional probability distributions, making it particularly useful in Bayesian inference and machine learning applications.

### **3.1 Why Use Langevin Dynamics?**
The intuition behind MALA comes from **Langevin diffusion**, a stochastic process that describes how particles move under **both deterministic forces (gradient of $ \log \pi(x) $) and stochastic noise**:

$$
dx = \frac{1}{2} \nabla \log \pi(x) \, dt + dW_t,
$$

where $ W_t $ is a **Wiener process** (Brownian motion). The key idea is:
- The **gradient term $ \nabla \log \pi(x) $** **pushes** the sample towards high-probability regions of $ \pi(x) $.
- The **stochastic term $ dW_t $** ensures sufficient exploration of the space.

By discretizing this continuous-time process using a small step size $ \epsilon $, we get the **Langevin proposal**:

$$
x' = x + \frac{\epsilon^2}{2} \nabla \log \pi(x) + \epsilon \eta, \quad \eta \sim \mathcal{N}(0,1).
$$

This proposal **biases the Markov chain toward regions of higher probability**, increasing sampling efficiency.

### **3.2 The Proposal Distribution and Acceptance Probability**
MALA modifies the **Metropolis-Hastings algorithm** by using a **Langevin-informed proposal**:

$$
q(x' \mid x) = \mathcal{N}\!\Bigl(x'; \, x + \frac{\epsilon^2}{2}\nabla \log \pi(x),\, \epsilon^2\Bigr).
$$

Unlike a standard **random walk proposal**, this Langevin proposal is **asymmetric**—it incorporates information about the target distribution.

Thus, the acceptance probability needs to correct for this asymmetry:

$$
\alpha(x, x') = \min\left\{1, \frac{\pi(x')\,q(x\mid x')}{\pi(x)\,q(x'\mid x)}\right\}.
$$

where $ q(y \mid x) $ is the transition probability:

$$
q(y \mid x) = \mathcal{N}\!\Bigl(y; \, x + \frac{\epsilon^2}{2}\nabla \log \pi(x),\, \epsilon^2\Bigr).
$$

This ensures that the Markov chain **converges to the correct stationary distribution $ \pi(x) $** while benefiting from gradient information.

### **3.3 Why Does MALA Work Better Than Metropolis-Hastings?**
**Key advantages of MALA over standard Metropolis-Hastings (MH):**
1. **Faster Convergence:**  
   - MH uses **uninformed proposals**, making random moves without knowledge of $ \pi(x) $.  
   - MALA **biases moves toward high-density regions**, reducing random rejections.
   
2. **Higher Acceptance Rate:**  
   - Standard MH can have a low acceptance rate if the proposal variance is too large.  
   - MALA **adapts** its proposal based on the local structure of $ \pi(x) $.

3. **Improved Exploration in High Dimensions:**  
   - In high-dimensional spaces, **random walk proposals** explore inefficiently.  
   - MALA **moves intelligently along probability gradients**.

However, **MALA also has limitations**:
- It requires **gradient calculations** ($\nabla \log \pi(x)$), which can be expensive.
- If the step size $ \epsilon $ is too large, the **proposal can be inaccurate**, leading to high rejection rates.

### **3.4 Efficiency Considerations**
MALA **balances exploration and exploitation** using the step size $ \epsilon $.

#### **How to choose $ \epsilon $ (step size)?**
- If $ \epsilon $ is **too small**:
  - Moves are **highly correlated**.
  - The chain **mixes slowly**, requiring more iterations.
- If $ \epsilon $ is **too large**:
  - The Langevin proposal **deviates too much** from the true probability flow.
  - The **acceptance rate drops**, leading to inefficient sampling.

#### **Rule of thumb:**
- **Tuning $ \epsilon $ to achieve an acceptance rate of ~50%** results in **optimal mixing**.
- Adaptive MCMC methods like **RMSprop MALA** or **preconditioned Langevin dynamics** can adjust $ \epsilon $ on the fly.

In [ ]:
def mala_sampling(target, step_size, num_samples=1000):
    """
    Perform sampling using the Metropolis-adjusted Langevin algorithm (MALA).
    
    Args:
        target (pyro.distributions.Distribution): The target distribution π(x).
        step_size (float): The step size (ε) for the Langevin proposal.
        num_samples (int): Number of samples to generate.
    
    Returns:
        list: Samples approximating the target distribution.
    """
    samples = []
    # Initialize the chain at 0.0 and enable gradient tracking.
    x = torch.tensor(0.0, requires_grad=True)
    
    for _ in range(num_samples):
        # Clear gradients 
        if x.grad is not None:
            x.grad.zero_()

        # Compute gradient of log π(x)
        log_prob = target.log_prob(x)
        log_prob.backward()
        grad_x = x.grad.detach()  # gradient ∇ log π(x)
        
        # Compute the proposal mean: x + (step_size²/2)*grad_x
        proposal_mean = x + (step_size**2 / 2) * grad_x
        
        # Propose new state: add Gaussian noise scaled by step_size
        noise = torch.randn(1) * step_size
        x_proposed = proposal_mean + noise
        
        # Compute the forward proposal log probability: q(x_proposed | x)
        log_q_forward = dist.Normal(x + (step_size**2/2)*grad_x, step_size).log_prob(x_proposed)
        
        # To compute the reverse proposal density q(x | x_proposed),
        # compute the gradient at x_proposed.
        x_proposed_temp = x_proposed.clone().detach().requires_grad_(True)
        log_prob_proposed = target.log_prob(x_proposed_temp)
        log_prob_proposed.backward()
        grad_x_proposed = x_proposed_temp.grad.detach()
        log_q_reverse = dist.Normal(x_proposed + (step_size**2/2)*grad_x_proposed, step_size).log_prob(x)
        
        # Compute the acceptance probability:
        # log_accept_ratio = log π(x_proposed) - log π(x) + log q(x|x_proposed) - log q(x_proposed|x)
        log_accept_ratio = target.log_prob(x_proposed) - target.log_prob(x) + log_q_reverse - log_q_forward
        accept_ratio = torch.exp(log_accept_ratio)
        
        # Accept or reject the proposal.
        if torch.rand(1).item() < min(1, accept_ratio.item()):
            # Accept the proposal
            x = x_proposed.detach().clone().requires_grad_(True)
        else:
            # Reject: keep the current state
            x = x.detach().clone().requires_grad_(True)
            
        samples.append(x.item())
    
    return samples

# Set the step size and number of samples for MALA
step_size = 0.1
num_samples = 1000

# Time the Langevin sampling procedure
ls_samples, ls_time = time_sampling(mala_sampling, target, step_size, num_samples=num_samples)
print(f"Langevin sampling (MALA) took {ls_time:.2f} seconds for {num_samples} samples.")

In [ ]:
# Plot the histogram of the Langevin samples with a kernel density estimate (KDE)
sns.histplot(ls_samples, kde=True)
plt.title("Langevin Sampling (MALA)")
plt.xlabel("x")
plt.ylabel("Frequency")
plt.show()

## 4. Hamiltonian Monte Carlo (HMC)

### **4.1 Why Use Hamiltonian Monte Carlo?**
**Hamiltonian Monte Carlo (HMC)** is an advanced **Markov Chain Monte Carlo (MCMC)** method that dramatically improves sampling efficiency, especially in **high-dimensional spaces**. Unlike Metropolis-Hastings and Langevin algorithms, which rely on local random-walk proposals, HMC **simulates physical dynamics** to propose distant yet probable states, leading to:
- **Better exploration** of the probability landscape.
- **Lower autocorrelation** between samples.
- **Faster convergence** to the true distribution.

HMC achieves this by introducing an **auxiliary momentum variable** and using **Hamiltonian dynamics** to make large moves in the state space while preserving a high acceptance rate.

### **4.2 The Hamiltonian Formulation**
HMC is inspired by **classical mechanics**, where a system is described by:
- **Position** $x$ (the parameter of interest).
- **Momentum** $p$ (an auxiliary variable that helps traverse the space).
- **Total energy** $H(x, p)$, known as the **Hamiltonian**.

The Hamiltonian function is defined as:

$$
H(x, p) = U(x) + K(p)
$$

where:
- $U(x) = -\log \pi(x)$ is the **potential energy**, derived from the target distribution.
- $K(p) = \frac{p^2}{2}$ is the **kinetic energy**, assuming $p \sim \mathcal{N}(0,1)$.

By introducing momentum variables, HMC transforms the sampling problem into a **dynamical system**, where updates follow a trajectory rather than a simple random walk.

### **4.3 Simulating Hamiltonian Dynamics**
Hamiltonian dynamics are governed by the equations:

$$
\frac{dx}{dt} = \frac{\partial H}{\partial p} = p, \quad \frac{dp}{dt} = -\frac{\partial H}{\partial x} = -\nabla U(x).
$$

Since solving these equations exactly is impractical, we use a **numerical integrator**, specifically the **Leapfrog method**, to approximate the evolution:

1. **Half-step momentum update**:  
   $$
   p \leftarrow p - \frac{\epsilon}{2} \nabla U(x).
   $$
2. **Full-step position update**:  
   $$
   x \leftarrow x + \epsilon p.
   $$
3. **Half-step momentum update**:  
   $$
   p \leftarrow p - \frac{\epsilon}{2} \nabla U(x).
   $$

where $\epsilon$ is the **step size**, controlling how far we move in each iteration.

After simulating this trajectory for **$L$ leapfrog steps**, the proposed state $(x', p')$ is accepted with probability:

$$
\alpha = \min\left(1, \frac{\exp(-H(x', p'))}{\exp(-H(x, p))}\right).
$$

Since Hamiltonian dynamics **preserve volume and energy**, proposals often have **high acceptance rates**, typically around 60–90%.

### **4.4 Why is HMC More Efficient?**

- **MH struggles in high dimensions** because random walks explore inefficiently.
- **MALA improves efficiency** by following gradients but still takes small steps.
- **HMC makes informed, long-distance jumps** while preserving the correct distribution, making it the best choice for large-scale Bayesian inference.


### **4.5 Tuning Hyperparameters in HMC**
HMC requires careful tuning of **step size** and **number of leapfrog steps** to optimize performance.

#### **(a) Step Size $\epsilon$**
- **Too small**: The chain moves slowly, requiring more steps.
- **Too large**: The leapfrog integrator accumulates errors, leading to low acceptance rates.

#### **(b) Number of Leapfrog Steps $L$**
- **Too few**: The proposals remain close to the current state, reducing efficiency.
- **Too many**: Computational overhead increases unnecessarily.


In [ ]:
def hmc_model():
    """
    A probabilistic model for HMC where x is drawn from a standard normal distribution.
    
    Returns:
        torch.Tensor: A sample from the target distribution.
    """
    # Sample x from the target distribution, $\pi(x) = \mathcal{N}(0,1)$
    return pyro.sample("x", dist.Normal(0, 1))

# Define the HMC kernel with a specified step size and number of leapfrog steps
hmc_kernel = mcmc.HMC(hmc_model, step_size=0.1, num_steps=10)

# Configure the MCMC run using HMC: 500 samples with 100 warm-up steps
mcmc_run = mcmc.MCMC(hmc_kernel, num_samples=500, warmup_steps=100)
mcmc_run.run()

# Extract the samples from the HMC chain for the variable 'x'
samples = mcmc_run.get_samples()["x"].cpu().numpy()

In [ ]:
# Plot the histogram of the HMC samples with KDE
sns.histplot(samples, kde=True)
plt.title("Hamiltonian Monte Carlo (HMC)")
plt.xlabel("x")
plt.ylabel("Frequency")
plt.show()

## **5. Markov Chain Monte Carlo (MCMC) using No-U-Turn Sampler (NUTS)**

### **5.1 Why Use NUTS?**
The **No-U-Turn Sampler (NUTS)** is an advanced extension of **Hamiltonian Monte Carlo (HMC)** that **automatically tunes hyperparameters**, making it one of the most efficient MCMC algorithms for high-dimensional Bayesian inference.

#### **Problems with Standard HMC**
- **Requires tuning of step size $\epsilon$ and number of leapfrog steps $L$**.
- **Too many leapfrog steps** → wasted computation.
- **Too few leapfrog steps** → inefficient exploration.

#### **How NUTS Solves This?**
NUTS **eliminates the need to set $L$ manually** by:
- **Automatically determining the optimal number of leapfrog steps**.
- **Stopping the trajectory when it starts to "double back" (U-turning)**, ensuring efficient exploration.

This **adaptive tuning** makes NUTS highly efficient, especially in **high-dimensional spaces**.

### **5.2 How NUTS Works**
NUTS follows the same **Hamiltonian dynamics** as HMC but incorporates an **adaptive stopping criterion**:

1. **Initialize position and momentum**:  
   - Start at $x_0$ and randomly draw momentum $p \sim \mathcal{N}(0,1)$.

2. **Simulate Hamiltonian dynamics bidirectionally**:
   - Instead of a fixed $L$, NUTS **doubles** the trajectory in both forward and backward directions until:
     - A "U-turn" is detected → the chain starts reversing direction.
     - The **tree-based expansion** balances efficiency with exploration.

3. **Accept or reject the proposal**:
   - Compute the Metropolis acceptance probability:
     $$
     \alpha = \min\left(1, \frac{\exp(-H(x', p'))}{\exp(-H(x, p))}\right).
     $$
   - If accepted, move to $x'$; otherwise, stay at $x$.

4. **Adapt step size $\epsilon$**:
   - During **warm-up**, NUTS tunes $\epsilon$ using **dual averaging**, optimizing exploration.


### **5.3. Key Advantages of NUTS**
- **No manual tuning**: Learns optimal parameters during warm-up.  
- **Eliminates inefficient leapfrog steps**: Stops automatically when needed.  
- **High acceptance rates (60–90%)**: Leads to better mixing.  
- **Handles high-dimensional problems**: Ideal for deep Bayesian models.  

In [ ]:
# Define the probabilistic model for NUTS sampling.
def model():
    """
    A simple probabilistic model where x is drawn from a standard normal distribution.
    
    Returns:
        torch.Tensor: A sample from the target distribution.
    """
    # Here, we assume that our variable x follows a standard normal distribution:
    # π(x) = Normal(0, 1)
    x = pyro.sample("x", dist.Normal(0, 1))
    return x

# Define the NUTS kernel using the model.
nuts_kernel = mcmc.NUTS(model)

# Configure the MCMC run
mcmc_run = mcmc.MCMC(nuts_kernel, num_samples=500, warmup_steps=100)

# Run the MCMC sampler
mcmc_run.run()

# Extract samples for the variable 'x' from the MCMC chain.
samples = mcmc_run.get_samples()["x"].cpu().numpy()

In [ ]:
sns.histplot(samples, kde=True)
plt.title("MCMC Sampling using NUTS")
plt.xlabel("x")
plt.ylabel("Frequency")
plt.show()

## **6. Summary of MCMC Methods**

We have explored several **Monte Carlo sampling algorithms** that allow us to draw samples from a **target distribution $\pi(x)$** when direct sampling is difficult. Each method has its strengths and weaknesses, making them suitable for different types of problems.

### **6.1 Key Differences and Trade-offs**
Each method **balances exploration and computational cost** differently:

- **Rejection Sampling** generates **independent** samples but can be inefficient in high dimensions.
- **Metropolis-Hastings (MH)** uses a **Markov Chain** but can struggle in high-dimensional spaces due to random-walk behavior.
- **Metropolis-Adjusted Langevin Algorithm (MALA)** improves upon MH by incorporating **gradient information** to make informed proposals.
- **Hamiltonian Monte Carlo (HMC)** introduces **momentum variables** to propose distant yet probable states, drastically improving efficiency.
- **No-U-Turn Sampler (NUTS)** further enhances HMC by **automatically adapting step sizes and trajectory lengths**, removing the need for manual tuning.

### **6.2 Comparison Table of MCMC Methods**
| **Method** | **Proposal Strategy** | **Uses Gradient?** | **Acceptance Rate** | **Exploration Efficiency** | **Best for High-Dimensional Problems?** |
|------------|----------------------|----------------|----------------|----------------------|--------------------------------|
| **Rejection Sampling** | Propose from $q(x)$, accept/reject | ❌ No | Varies (low if $M$ is large) | Poor (too many rejections) | ❌ No |
| **Metropolis-Hastings (MH)** | Random walk $q(x' \mid x)$ | ❌ No | Moderate (~20-40%) | Poor (random-walk behavior) | ❌ No |
| **MALA (Langevin Sampling)** | Gradient-informed proposal | ✅ Yes ($\nabla \log \pi(x)$) | Moderate (~40-60%) | Good | ✅ Yes (but still local moves) |
| **HMC (Hamiltonian Monte Carlo)** | Simulated Hamiltonian dynamics | ✅ Yes ($\nabla \log \pi(x)$) | High (~60-90%) | Excellent (large jumps) | ✅ Yes |
| **NUTS (No-U-Turn Sampler)** | Adaptive Hamiltonian dynamics | ✅ Yes ($\nabla \log \pi(x)$) | Very High (~70-95%) | Best (auto-tuned efficiency) | ✅✅ **Yes (best choice!)** |
